In [1]:
from pathlib import Path
import sys

BASE_DIR = Path().cwd().parent

PICTURE_DATA_DIR = BASE_DIR / "picture_data"
DATASET_DIR = PICTURE_DATA_DIR / "dataset"
TRAIN_DIR = DATASET_DIR / "train"
VALIDATION_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"

FINE_TUNING_DIR = BASE_DIR / "fine_tuning"
FEATURE_DIR = FINE_TUNING_DIR / "features"
MODEL_DIR = FINE_TUNING_DIR / "models"

CONFUSED_IMAGES_DIR = BASE_DIR / "confused_images"

VERSION = 2
VARIANT = 3

CHECKPOINT_PATH = MODEL_DIR / f"fine_tuned_base_{VARIANT}.keras"

# 1: Basis
# 2: Mehr Daten, fix Doppeldeutigkeiten
# 3: + Dropout, mehr Daten (Brille)

CLASS_NAMES = [
    "brille",
    "flasche",
    "geldboerse",
    "handy",
    "hund",
    "kirche",
    "schuh",
    "tasse"
]

sys.path.append(str(BASE_DIR))

from data_handling.manage_data import create_dataset, load_features
from data_handling.create_confusion_images import export_misclassified_images
from model.build_model import save_model, load_model, build_finetuning_model
from evaluation.evaluation import evaluate_model

from model.training_object_classes import EpochInfo, get_early_stopping, get_checkpoint

In [2]:
def fine_tune ():
    model = build_finetuning_model(dropout=0.3)

    model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=30,
        shuffle=True,
        callbacks=[
            EpochInfo(), 
            get_early_stopping(patience=3, min_delta=0),
            get_checkpoint(CHECKPOINT_PATH)
        ],
        verbose=1
    )

In [3]:
X_train, y_train, X_validation, y_validation, X_test, y_test = load_features(FEATURE_DIR, VERSION)

In [4]:
fine_tune()

Epoch 1/30
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 295ms/step - accuracy: 0.5152 - loss: 1.4671
Epoche 1:  53.02 Sekunden
System RAM: 75.3 %
System verfügbar: 3.42 GB
Python Prozess: 4.71 GB

159/159 ━━━━━━━━━━━━━━━━━━━━ 54s 315ms/step - accuracy: 0.5152 - loss: 1.4671 - val_accuracy: 0.8356 - val_loss: 0.5228
Epoch 2/30
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step - accuracy: 0.8573 - loss: 0.4407
Epoche 2:  47.72 Sekunden
System RAM: 76.7 %
System verfügbar: 3.22 GB
Python Prozess: 4.74 GB

159/159 ━━━━━━━━━━━━━━━━━━━━ 48s 302ms/step - accuracy: 0.8573 - loss: 0.4407 - val_accuracy: 0.8742 - val_loss: 0.3502
Epoch 3/30
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 284ms/step - accuracy: 0.9148 - loss: 0.2604
Epoche 3:  46.86 Sekunden
System RAM: 77.0 %
System verfügbar: 3.18 GB
Python Prozess: 4.74 GB

159/159 ━━━━━━━━━━━━━━━━━━━━ 47s 297ms/step - accuracy: 0.9148 - loss: 0.2604 - val_accuracy: 0.8993 - val_loss: 0.3325
Epoch 4/30
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step - accuracy: 0.9431 - loss: 0.1652


In [5]:
model = load_model(MODEL_DIR, f"fine_tuned_base_{VARIANT}", compile=False)

c:\Users\MarcelKramer\Desktop\Unterrichtsmaterialien\08. Machine Learning\Abschlussprojekt\cnn_env\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 6 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(store)


In [6]:
evaluation, result_df = evaluate_model(model, X_validation, y_validation, CLASS_NAMES)

evaluation

19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 109ms/step


{'accuracy': 0.9026845637583892,
 'balanced_accuracy': 0.8963672917093,
 'log_loss': 0.2940954864025116,
 'confusion_matrix': array([[66,  0,  0,  4,  0,  0,  4,  0],
        [ 2, 94,  0,  6,  3,  0,  2,  1],
        [ 3,  1, 32,  6,  0,  0,  0,  0],
        [ 2,  1,  3, 60,  0,  0,  0,  0],
        [ 0,  1,  0,  1, 52,  0,  2,  1],
        [ 0,  1,  0,  0,  0, 72,  0,  0],
        [ 2,  0,  2,  0,  1,  0, 87,  1],
        [ 3,  3,  0,  1,  0,  0,  1, 75]]),
 'classification_report': {'brille': {'precision': 0.8461538461538461,
   'recall': 0.8918918918918919,
   'f1-score': 0.868421052631579,
   'support': 74.0},
  'flasche': {'precision': 0.9306930693069307,
   'recall': 0.8703703703703703,
   'f1-score': 0.8995215311004785,
   'support': 108.0},
  'geldboerse': {'precision': 0.8648648648648649,
   'recall': 0.7619047619047619,
   'f1-score': 0.810126582278481,
   'support': 42.0},
  'handy': {'precision': 0.7692307692307693,
   'recall': 0.9090909090909091,
   'f1-score': 0.83333333

In [7]:
validation_images = create_dataset(VALIDATION_DIR, class_names=CLASS_NAMES, is_preprocess=False).file_paths

Found 596 files belonging to 8 classes.


In [8]:
result_df = result_df.reset_index()

result_df["image_path"] = result_df["index"].apply(lambda x: validation_images[x])

result_df

confusion_filter = result_df["y_true"] != result_df["y_pred"]

confusion_df = result_df[confusion_filter]

confusion_df

,index,y_true,y_pred,y_proba,image_path
14,14,brille,schuh,0.468649,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
15,15,brille,handy,0.617796,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
18,18,brille,schuh,0.982896,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
28,28,brille,handy,0.662911,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
29,29,brille,handy,0.387581,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
45,45,brille,schuh,0.648816,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
57,57,brille,schuh,0.493086,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
58,58,brille,handy,0.348496,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
77,77,flasche,schuh,0.886651,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...
78,78,flasche,hund,0.359726,c:\Users\MarcelKramer\Desktop\Unterrichtsmater...


In [9]:
print(confusion_df.groupby(["y_true", "y_pred"]).size().sort_values(ascending=False))

y_true      y_pred    
geldboerse  handy         6
flasche     handy         6
brille      schuh         4
            handy         4
geldboerse  brille        3
flasche     hund          3
tasse       flasche       3
            brille        3
handy       geldboerse    3
flasche     schuh         2
schuh       geldboerse    2
            brille        2
flasche     brille        2
handy       brille        2
hund        schuh         2
flasche     tasse         1
geldboerse  flasche       1
handy       flasche       1
kirche      flasche       1
hund        tasse         1
            handy         1
            flasche       1
schuh       tasse         1
            hund          1
tasse       handy         1
            schuh         1
dtype: int64


In [10]:
export_misclassified_images(confusion_df, CONFUSED_IMAGES_DIR, CLASS_NAMES)